# Chapter 1 — What Is Generative AI? (hands-on)

This notebook turns the ideas from Chapter 1 into things you can **run and see**. It is written
for network engineers who have **never written code or used AI before**. You do not need to
understand every line — just read the note above each cell, press the ▶ button, and watch what
happens.

What you will do:
1. Connect to the AI (Claude) — like logging into a new system for the first time
2. Ask it a networking question
3. Give it a router config and ask for a review
4. See how "tokens" decide what each request costs
5. Compare the old way (regex) with the AI way
6. Learn to remove passwords before sending a config anywhere

**Time:** ~15 minutes. **Cost:** about 10–20 cents of API usage. **Setup:** none — it all runs
in your browser.

> Run the cells **in order, top to bottom.** Each one builds on the cell before it.

## Before you start — get an API key

An **API key** is your password to the AI service (the chapter compares it to TACACS+
credentials). You get one from https://console.anthropic.com/ → *API Keys* → *Create Key*. It
looks like `sk-ant-...`.

**The safe way to use it in Colab** (so it is not visible in the notebook):
1. Click the **🔑 key icon** in the left sidebar of Colab ("Secrets").
2. Add a new secret named exactly `ANTHROPIC_API_KEY` and paste your key as the value.
3. Turn on "Notebook access."

If you skip that, the next cell will simply ask you to type the key (it stays hidden). Never
paste your key directly into a code cell you might share.

## Cell 1 — Connect to the AI

This cell does three small jobs:
- **Installs** the `anthropic` library — the piece of software that knows how to talk to Claude.
- **Loads your API key** (from Colab Secrets if you set it up, otherwise it asks you to type it).
- **Creates `client`** — think of this as your open SSH session to the AI. Every later cell uses it.

`MODEL` picks which AI to use. We use **Haiku**, the fastest and cheapest one, which is perfect
for learning. (Note: the name uses dashes — `claude-haiku-4-5`. A dotted `4.5` would error.)

In [1]:
!pip install -q anthropic

import os
from anthropic import Anthropic

# Get the API key from Colab Secrets if available, otherwise ask for it.
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")

client = Anthropic()                  # opens the connection (reads the key from the environment)
MODEL  = "claude-haiku-4-5"           # fast + cheap, great for learning
print("Connected. Ready to ask the AI questions.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 11.2 MB/s eta 0:00:00


TypeError: str expected, not dict

## Cell 2 — Ask your first question

This is the whole idea of an AI API in one cell: you **send some text** (called a *prompt*) and
you **get text back**. Here we ask Claude to explain OSPF.

A few words you will keep seeing:
- **prompt** — the question or task you send (the `content` below).
- **`max_tokens`** — a limit on how long the answer can be (a safety cap, not a target).
- **`response.content[0].text`** — where the AI's written answer lives.
- **tokens** — the AI measures text in tokens (~¾ of a word). The last line shows how many you
  used; that is what you are charged for.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": "Explain what OSPF is in 3 sentences, as if explaining to a junior network engineer."
    }]
)

print(response.content[0].text)
print(f"\nTokens used: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

## Cell 3 — Give it a config to review

Now something genuinely useful. We hand Claude a router configuration and ask it to review it,
the way a senior engineer would. Notice we add a **`system`** message — a standing instruction
that sets the AI's role ("you are a senior network engineer"). The chapter compares this to a
route-map applied to everything.

Watch for this: `GigabitEthernet0/1` is **shut down** but is still inside the OSPF network
range. You did not write a rule to catch that — the AI notices it on its own.

In [ ]:
sample_config = '''
hostname router-core-01
!
interface Loopback0
 ip address 192.168.1.1 255.255.255.255
!
interface GigabitEthernet0/0
 description Uplink to ISP-A
 ip address 203.0.113.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description Trunk to DC-SPINE-01
 ip address 10.0.1.1 255.255.255.0
 shutdown
!
router ospf 1
 router-id 192.168.1.1
 network 10.0.0.0 0.0.255.255 area 0
!
ip route 0.0.0.0 0.0.0.0 203.0.113.2
'''

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="You are a senior network engineer performing a configuration review.",
    messages=[{
        "role": "user",
        "content": f'''Review this router configuration and identify:
1. Security concerns
2. Best practices followed
3. Anything misconfigured or suspicious

Configuration:
{sample_config}'''
    }]
)

print(response.content[0].text)

## Cell 4 — Tokens and cost (the "bandwidth budget" of AI)

The chapter's key idea: **tokens are the packets of AI.** You pay per token, and each request
has a maximum it can hold. This cell uses `count_tokens` to measure the config *before* sending
it, then estimates the cost with two different models.

The numbers are tiny here (fractions of a cent), but multiply by 500 devices and it adds up —
which is why you pick a cheaper model (Haiku) for simple, bulk work.

> Prices change. Check the current rates at https://www.anthropic.com/pricing before you rely on
> a number. The values below are the per-token rates the chapter quotes.

In [ ]:
count = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": sample_config}]
)

PRICE_HAIKU_PER_TOKEN  = 0.0000008    # $0.80 per million input tokens
PRICE_SONNET_PER_TOKEN = 0.000003     # $3.00 per million input tokens

print(f"This config is {count.input_tokens} tokens.")
print(f"Cost to send it once with Haiku : ${count.input_tokens * PRICE_HAIKU_PER_TOKEN:.6f}")
print(f"Cost to send it once with Sonnet: ${count.input_tokens * PRICE_SONNET_PER_TOKEN:.6f}")

## Cell 5 — The old way vs the AI way

Same job — pull the interfaces out of a config — done two ways:
- **Regex** (traditional code): fast and exact, but you wrote it for Cisco IOS. Point it at
  Juniper or Arista and it breaks.
- **AI**: you just *describe* what you want, and the same request works across vendors.

Neither is "better" everywhere. The chapter's rule: regex when you need 100% certainty on one
vendor; AI when you need flexibility across many. Run it and compare the two outputs.

In [ ]:
import re, json

# --- Old way: a regex parser (tuned for Cisco IOS) ---
def parse_interfaces_regex(config):
    interfaces = []
    for m in re.finditer(r'interface (\S+)\n((?:[ !].*\n)*)', config, re.MULTILINE):
        name, body = m.group(1), m.group(2)
        ip   = re.search(r'ip address (\S+)', body)
        desc = re.search(r'description (.+)', body)
        interfaces.append({
            "name": name,
            "ip": ip.group(1) if ip else None,
            "description": desc.group(1).strip() if desc else None,
            "shutdown": "shutdown" in body and "no shutdown" not in body,
        })
    return interfaces

# --- AI way: just ask (works for any vendor) ---
def parse_interfaces_ai(config):
    resp = client.messages.create(
        model=MODEL, max_tokens=1024,
        messages=[{"role": "user", "content":
            f'''Extract all interfaces from this config.
Return ONLY a JSON list with fields: name, ip_address, description, is_shutdown.

Config:
{config}'''}]
    )
    text = resp.content[0].text
    return json.loads(text[text.find('['): text.rfind(']') + 1])

print("=== Old way (regex) ===")
print(parse_interfaces_regex(sample_config))
print("\n=== AI way ===")
print(parse_interfaces_ai(sample_config))

## Cell 6 — Remove secrets before sending (do this every time)

When you send a config to any cloud AI, it leaves your network. So **strip the secrets first** —
passwords, SNMP community strings, keys. This cell hides them with a few find-and-replace rules,
then shows the cleaned config. Send the *cleaned* version, never the raw one.

(For very sensitive environments, the chapter suggests running a model on your own servers so no
data leaves at all.)

In [ ]:
import re

def sanitize_config(config: str) -> str:
    '''Hide passwords, community strings and keys before sending anywhere.'''
    rules = [
        (r'(password|secret)\s+\S+',               r'\1 ***REDACTED***'),
        (r'(snmp-server community)\s+\S+',          r'\1 ***REDACTED***'),
        (r'(key)\s+\S+',                            r'\1 ***REDACTED***'),
    ]
    for pattern, replacement in rules:
        config = re.sub(pattern, replacement, config, flags=re.IGNORECASE)
    return config

raw = '''interface Tunnel0
 ip address 10.0.0.1 255.255.255.0
snmp-server community S3cr3tString RO
username admin password MyPassw0rd
'''

print("BEFORE (do NOT send this):\n" + raw)
print("AFTER (safe to send):\n" + sanitize_config(raw))

## You're done — what you just learned

- An AI API is simple: **send text, get text back** — the same shape as any REST API.
- The AI can **review configs and find problems** you did not explicitly tell it to look for.
- **Tokens** are the unit of size and cost — budget them like bandwidth.
- Use **regex for one vendor with exact rules**, **AI for flexibility across vendors**.
- Always **remove secrets** before sending a config to a cloud AI.

**The golden rule from the chapter:** the AI is a very fast, very knowledgeable colleague — but
it can be confidently wrong ("hallucinate"). **Never apply an AI-generated config to production
without a human review.** Trust, but verify.

Next: try changing the prompts above — paste one of *your own* (sanitized!) configs into Cell 3
and see what the review finds.